[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jairomelo/aiOCR/blob/main/models/SmolVLM2-2B/SmolVLM2-2B.ipynb)

## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# SmolVLM2 requires transformers >= 4.49.0 (AutoModelForVision2Seq added there).
# Pin below 5.0: transformers 5.0 removed AutoModelForVision2Seq.
!pip install "transformers>=5.0" accelerate Pillow num2words

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 23.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [3]:
# SmolVLM2-2B-Instruct is a gated repo — authenticate with your HF token.
# In Colab: Secrets (🔑 left panel) → add HF_TOKEN with your token from huggingface.co/settings/tokens
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [4]:
import transformers
print(f"transformers {transformers.__version__}")

from packaging.version import Version
if Version(transformers.__version__) < Version("5.0"):
    print("Version too old — restarting runtime...")
    import os; os.kill(os.getpid(), 9)
else:
    print("OK — continue to the next cell.")

transformers 5.6.2
OK — continue to the next cell.


In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText

In [6]:
import torch

In [7]:
from pathlib import Path
from PIL import Image

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'HuggingFaceTB/SmolVLM2-2.2B-Instruct'

# SmolVLM2-2.2B: SigLIP vision encoder + SmolLM2-2.2B decoder.
# ~4.4 GB fp16 — no quantization needed on T4.
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
).eval().cuda()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [8]:
IMAGE_FILES = [
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_1.png',
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_46.png',
    WORKING_DIR / 'images/CO_18180627/CO_18180627_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_4.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_1.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_2.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_1.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_3.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_4.png',
    WORKING_DIR / 'images/AR_SR8V4R3/AR_SR8V4R3_4.jpg',
]

## Inference

In [9]:
import time

transcription_out = WORKING_DIR / 'transcriptions/SmolVLM2-2B'
transcription_out.mkdir(parents=True, exist_ok=True)

prompt = (
    'Convert the document to plain text, as close to the original as possible '
    '(including typos, print errors, and original grammar and spelling). '
    'Do not add any formatting, markdown, or annotations.'
)

for IMAGE_FILE in IMAGE_FILES:
    image_stem = IMAGE_FILE.stem
    out_path = transcription_out / f'{image_stem}.md'

    if out_path.exists():
        print(f'Skipping (already done): {image_stem}')
        continue

    if not IMAGE_FILE.exists():
        print(f'Skipping (image not found): {IMAGE_FILE}')
        continue

    image = Image.open(IMAGE_FILE).convert('RGB')

    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=text, images=[image], return_tensors='pt'
    ).to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False,
        )
    elapsed = time.time() - t0

    generated_ids_trimmed = [out[input_len:] for out in generated_ids]
    transcription = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
    )[0]

    out_path.write_text(transcription, encoding='utf-8')
    print(f'Done in {elapsed:.1f}s — saved: transcriptions/SmolVLM2-2B/{image_stem}.md')

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Done in 4.1s — saved: transcriptions/SmolVLM2-2B/CCundinamarca_page_1.md


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Done in 136.8s — saved: transcriptions/SmolVLM2-2B/CCundinamarca_page_46.md


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

Done in 2.5s — saved: transcriptions/SmolVLM2-2B/pineda1_page_3.md


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Done in 134.4s — saved: transcriptions/SmolVLM2-2B/pineda1_page_4.md


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Done in 2.1s — saved: transcriptions/SmolVLM2-2B/AR_SR8V4R3_4.md


### Saving the output